# Integration Weight Verification

This notebook comprehensively verifies that the quadrature weights generated by both Table 1 and Table 2 node generators sum correctly to 1.0 across all supported polynomial degrees (k=1,2,3,4).

**Purpose:** Validate the numerical correctness of the weighted quadrature rules before using them for modal expansion and integration.

**Expected Result:** All weights should sum to exactly 1.0 (within floating-point tolerance of 1e-14).

## Setup: Import Required Modules

In [1]:
import sys
from pathlib import Path
import math

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.generators import build_nodes
from src.core.render_utils import VERTICES_2D, VERTICES_3D


def get_reference_data(method: str, k: int):
    nodes = build_nodes(method, k, VERTICES_2D, VERTICES_3D)
    bary_coords = np.array([n.barycentric for n in nodes], dtype=float)
    weights = np.array([n.weight for n in nodes], dtype=float)
    xi = 2.0 * bary_coords[:, 2] - 1.0
    eta = 2.0 * bary_coords[:, 1] - 1.0
    return {
        "nodes": nodes,
        "bary_coords": bary_coords,
        "weights": weights,
        "xi": xi,
        "eta": eta,
    }


def exact_reference_integral(i: int, j: int) -> float:
    def int_sk(k):
        return 2.0 / (k + 1) if k % 2 == 0 else 0.0
    term1 = int_sk(i + j + 1)
    term2 = int_sk(j)
    sign = -1.0 if (i + 1) % 2 != 0 else 1.0
    return (sign / (2.0 * (i + 1))) * (term1 - term2)


def generate_exponent_tuples(degree: int):
    return [(i, degree - i) for i in range(degree + 1)]


def test_polynomial_exactness(xi: np.ndarray, eta: np.ndarray, weights: np.ndarray, max_degree: int, tolerance: float = 1e-13) -> pd.DataFrame:
    rows = []

    for degree in range(max_degree + 1):
        for i, j in generate_exponent_tuples(degree):
            exact_value = exact_reference_integral(i, j)

            numerical_value = np.sum(weights * (xi ** i) * (eta ** j))

            absolute_error = abs(numerical_value - exact_value)
            is_exact = np.isclose(numerical_value, exact_value, atol=tolerance)

            rows.append({
                "degree": degree,
                "term": f"xi^{i} * eta^{j}",
                "exact_value": exact_value,
                "numerical_value": numerical_value,
                "absolute_error": absolute_error,
                "status": "✓ PASS" if is_exact else "✗ FAIL",
            })

    return pd.DataFrame(rows)

## Comprehensive Weight Verification

Systematically iterate through all combinations of methods and polynomial degrees, validate weights, and generate a summary report.

In [2]:
METHODS = ["table1", "table2"]
K_VALUES = [1, 2, 3, 4]
TOLERANCE = 1e-14

rows = []

for method in METHODS:
    for k in K_VALUES:
        try:
            ref = get_reference_data(method, k)
            weights = ref["weights"]
            weight_sum = float(np.sum(weights))
            error = abs(weight_sum - 1.0)
            is_valid = bool(np.isclose(weight_sum, 1.0, atol=TOLERANCE))

            rows.append({
                "method": method,
                "k": k,
                "num_nodes": len(ref["nodes"]),
                "weight_sum": weight_sum,
                "abs_error": error,
                "status": "✓ PASS" if is_valid else "✗ FAIL",
                "is_valid": is_valid,
            })
        except Exception as exc:
            rows.append({
                "method": method,
                "k": k,
                "num_nodes": -1,
                "weight_sum": np.nan,
                "abs_error": np.nan,
                "status": f"✗ ERROR: {exc}",
                "is_valid": False,
            })

weights_df = pd.DataFrame(rows)
display(weights_df)

,method,k,num_nodes,weight_sum,abs_error,status,is_valid
0,table1,1,6,1.0,1.110223e-16,✓ PASS,True
1,table1,2,10,1.0,0.000000e+00,✓ PASS,True
2,table1,3,18,1.0,2.220446e-16,✓ PASS,True
3,table1,4,22,1.0,2.220446e-16,✓ PASS,True
4,table2,1,3,1.0,0.000000e+00,✓ PASS,True
5,table2,2,6,1.0,1.110223e-16,✓ PASS,True
6,table2,3,12,1.0,2.220446e-16,✓ PASS,True
7,table2,4,16,1.0,0.000000e+00,✓ PASS,True


## Summary Report

In [3]:
summary_df = (
    weights_df.groupby(["method", "status"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
display(summary_df)

all_passed = bool(weights_df["is_valid"].all())

if all_passed:
    print("✓ SUCCESS! All quadrature weight sums are valid (≈ 1.0 within tolerance 1e-14).")
else:
    print("✗ FAILURE! Some quadrature weight tests failed.")

failed_weights_df = weights_df.loc[~weights_df["is_valid"]].copy()
if not failed_weights_df.empty:
    display(failed_weights_df)

,method,status,count
0,table1,✓ PASS,4
1,table2,✓ PASS,4


✓ SUCCESS! All quadrature weight sums are valid (≈ 1.0 within tolerance 1e-14).


## Detailed Analysis (Optional)

Select a specific configuration to examine in detail:

In [4]:
EXAMINE_METHOD = "table1"
EXAMINE_K = 4

ref = get_reference_data(EXAMINE_METHOD, EXAMINE_K)
nodes = ref["nodes"]
weights = ref["weights"]
barycentric_coords = ref["bary_coords"]

detail_df = pd.DataFrame({
    "node_id": [n.node_id for n in nodes],
    "b1": barycentric_coords[:, 0],
    "b2": barycentric_coords[:, 1],
    "b3": barycentric_coords[:, 2],
    "weight": weights,
})
display(detail_df)

weight_stats_df = pd.DataFrame([
    {
        "method": EXAMINE_METHOD,
        "k": EXAMINE_K,
        "num_nodes": len(nodes),
        "weight_min": float(np.min(weights)),
        "weight_max": float(np.max(weights)),
        "weight_mean": float(np.mean(weights)),
        "weight_std": float(np.std(weights)),
        "weight_sum": float(np.sum(weights)),
        "abs_error_from_1": float(abs(np.sum(weights) - 1.0)),
    }
])
display(weight_stats_df)

,node_id,b1,b2,b3,weight
0,1,0.046910,0.000000,0.953090,0.006601
1,2,0.046910,0.953090,0.000000,0.006601
2,3,0.000000,0.046910,0.953090,0.006601
3,4,0.000000,0.953090,0.046910,0.006601
4,5,0.953090,0.046910,0.000000,0.006601
5,6,0.953090,0.000000,0.046910,0.006601
6,7,0.230765,0.000000,0.769235,0.020530
7,8,0.230765,0.769235,0.000000,0.020530
8,9,0.000000,0.230765,0.769235,0.020530
9,10,0.000000,0.769235,0.230765,0.020530


,method,k,num_nodes,weight_min,weight_max,weight_mean,weight_std,weight_sum,abs_error_from_1
0,table1,4,22,0.006601,0.124737,0.045455,0.046068,1.0,2.220446e-16


## Polynomial Exactness Test

Rigorously test that the quadrature rules exactly integrate monomials in 2D reference coordinates $(\xi, \eta)$. To match the orthogonal polynomial domain $T^2$ defined in the paper, we map from barycentric coordinates to the $[-1, 1]$ range by:

$$\xi = 2b_3 - 1, \quad \eta = 2b_2 - 1$$

Based on the simplex quadrature rule, the numerical integration is evaluated as a discrete sum over the nodes:

$$\int_{T} f(\xi, \eta) \, d\xi = |T| \sum_{m=1}^{N} f(\xi_m, \eta_m) w_m^s$$

Since the quadrature weights $w_m^s$ in this notebook are normalized to sum to 1.0 (implicitly dividing by the area $|T|$), and our test function is the monomial $f(\xi, \eta) = \xi^i \eta^j$, we evaluate the numerical summation $\sum w_m (\xi_m)^i (\eta_m)^j$.

This numerical value is then rigorously verified to machine precision against the exact normalized integral over the reference triangle $T^2$ (with vertices at $(-1,1), (-1,-1), (1,-1)$):

$$\frac{1}{\mathrm{Area}(T^2)} \iint_{T^2} \xi^i \eta^j \, d\xi \, d\eta$$

In [5]:
TOLERANCE_EXACTNESS = 1e-13
poly_rows = []

for method in METHODS:
    for k in K_VALUES:
        max_test_degree = (2 * k - 1) if method == "table1" else (2 * k)
        ref = get_reference_data(method, k)

        per_case_df = test_polynomial_exactness(
            xi=ref["xi"],
            eta=ref["eta"],
            weights=ref["weights"],
            max_degree=max_test_degree,
            tolerance=TOLERANCE_EXACTNESS,
        )

        per_case_df.insert(0, "method", method)
        per_case_df.insert(1, "k", k)
        poly_rows.append(per_case_df)

polynomial_exactness_df = pd.concat(poly_rows, ignore_index=True)
display(polynomial_exactness_df)

poly_summary_df = (
    polynomial_exactness_df.groupby(["method", "k", "status"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
display(poly_summary_df)

overall_exactness_passed = bool(polynomial_exactness_df["status"].eq("✓ PASS").all())
if overall_exactness_passed:
    print("✓ SUCCESS! All xi/eta polynomial exactness tests passed within tolerance.")
else:
    print("✗ FAILURE! Some xi/eta polynomial exactness tests failed.")
    failed_poly_df = polynomial_exactness_df.loc[~polynomial_exactness_df["is_exact"]].copy()
    display(failed_poly_df)

,method,k,degree,term,exact_value,numerical_value,absolute_error,status
0,table1,1,0,xi^0 * eta^0,1.000000,1.000000e+00,1.110223e-16,✓ PASS
1,table1,1,1,xi^0 * eta^1,-0.333333,-3.333333e-01,0.000000e+00,✓ PASS
2,table1,1,1,xi^1 * eta^0,-0.333333,-3.333333e-01,5.551115e-17,✓ PASS
3,table1,2,0,xi^0 * eta^0,1.000000,1.000000e+00,0.000000e+00,✓ PASS
4,table1,2,1,xi^0 * eta^1,-0.333333,-3.333333e-01,0.000000e+00,✓ PASS
...,...,...,...,...,...,...,...,...
159,table2,4,8,xi^4 * eta^4,0.040000,4.000000e-02,1.387779e-17,✓ PASS
160,table2,4,8,xi^5 * eta^3,0.000000,6.071532e-18,6.071532e-18,✓ PASS
161,table2,4,8,xi^6 * eta^2,0.047619,4.761905e-02,1.387779e-17,✓ PASS
162,table2,4,8,xi^7 * eta^1,0.000000,1.040834e-17,1.040834e-17,✓ PASS


,method,k,status,count
0,table1,1,✓ PASS,3
1,table1,2,✓ PASS,10
2,table1,3,✓ PASS,21
3,table1,4,✓ PASS,36
4,table2,1,✓ PASS,6
5,table2,2,✓ PASS,15
6,table2,3,✓ PASS,28
7,table2,4,✓ PASS,45


✓ SUCCESS! All xi/eta polynomial exactness tests passed within tolerance.
